# Modeling

Goal: baseline, XGBoost, LightGBM, Isolation Forest, SHAP

In [5]:
import pandas as pd

df=pd.read_parquet("D:/Coding/fraud-detection-realtime/data/processed/cleaned_data.parquet")

In [6]:
df.shape

(3000000, 26)

In [7]:
df.columns

Index(['user', 'card', 'year', 'month', 'day', 'time', 'amount', 'use_chip',
       'merchant_name', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors', 'is_fraud', 'timestamp', 'hour', 'outside_us', 'seen_before',
       'new_shop', 'new_shop_type', 'gap', 'gap_minutes', 'count_last_hour',
       'amount_vs_usual', 'share_same_hour'],
      dtype='str')

In [8]:
cutoff = df["timestamp"].quantile(0.8)
cutoff

Timestamp('2017-04-25 21:24:36')

In [9]:
train=df[df["timestamp"]<cutoff]
valid=df[df["timestamp"]>=cutoff]

In [10]:
print(train.shape, valid.shape)
print(train["is_fraud"].sum(), valid["is_fraud"].sum())

(2400000, 26) (600000, 26)
2805 616


In [11]:
features=["amount","hour","outside_us","new_shop","gap_minutes"]

In [12]:
X_train=train[features]
y_train=train["is_fraud"]

X_valid=valid[features]
y_valid=valid["is_fraud"]

In [13]:
X_train.isna().sum()

amount           0
hour             0
outside_us       0
new_shop         0
gap_minutes    545
dtype: int64

In [14]:
X_train = X_train.fillna({"gap_minutes": 100000})
X_valid = X_valid.fillna({"gap_minutes": 100000})

In [15]:
X_train.isna().sum().sum()
X_valid.isna().sum().sum()

np.int64(0)

In [16]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [17]:
model=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=1000))
model.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['amount','hour','outside_us','new_shop','gap_minutes']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [18]:
scores=model.predict_proba(X_valid)[:,1]
from sklearn.metrics import average_precision_score
print("PR-AUC:",average_precision_score(y_valid,scores))

PR-AUC: 0.0903578015543146


In [19]:
top=scores.argsort()[::-1][:1000]

In [20]:
y_valid.iloc[top].sum()

np.int64(150)

In [21]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
features = ["amount", "hour", "outside_us", "new_shop", "gap_minutes", "use_chip", "mcc"]

X_train = train[features].fillna({"gap_minutes": 100000})
X_valid = valid[features].fillna({"gap_minutes": 100000})

prep = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'),["use_chip","mcc"]),
    (StandardScaler(),["amount","hour","outside_us","new_shop","gap_minutes"])
)


In [22]:
model2=make_pipeline(prep,LogisticRegression(class_weight="balanced",max_iter=1000))
model2.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['amount','hour','outside_us',...,'gap_minutes','use_chip','mcc']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...), ('standardscaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [23]:
scores2 = model2.predict_proba(X_valid)[:, 1]
print("PR-AUC:", average_precision_score(y_valid, scores2))

PR-AUC: 0.07476199090348372


In [24]:
top2 = scores2.argsort()[::-1][:1000]
print("Frauds in top 1000:", y_valid.iloc[top2].sum())

Frauds in top 1000: 142


In [25]:
features3 = ["amount", "hour", "outside_us", "new_shop", "gap_minutes", "use_chip"]

X_train3 = train[features3].fillna({"gap_minutes": 100000})
X_valid3 = valid[features3].fillna({"gap_minutes": 100000})

prep3 = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), ["use_chip"]),
    (StandardScaler(), ["amount", "hour", "outside_us", "new_shop", "gap_minutes"])
)

model3 = make_pipeline(prep3, LogisticRegression(class_weight="balanced", max_iter=1000))
model3.fit(X_train3, y_train)

scores3 = model3.predict_proba(X_valid3)[:, 1]
print("PR-AUC:", average_precision_score(y_valid, scores3))
top3 = scores3.argsort()[::-1][:1000]
print("Frauds in top 1000:", y_valid.iloc[top3].sum())

PR-AUC: 0.041118118545853
Frauds in top 1000: 17


In [26]:
print(train["use_chip"].value_counts(normalize=True)*100)
print(valid["use_chip"].value_counts(normalize=True)*100)

use_chip
Swipe Transaction     74.232875
Chip Transaction      13.526167
Online Transaction    12.240958
Name: proportion, dtype: float64
use_chip
Chip Transaction      68.507833
Swipe Transaction     17.541833
Online Transaction    13.950333
Name: proportion, dtype: float64


In [27]:
print(train.groupby("use_chip")["is_fraud"].mean() * 10000)
print(valid.groupby("use_chip")["is_fraud"].mean() * 10000)

use_chip
Chip Transaction       2.556773
Online Transaction    66.375522
Swipe Transaction      4.333210
Name: is_fraud, dtype: float64
use_chip
Chip Transaction      12.504653
Online Transaction     0.000000
Swipe Transaction      9.691119
Name: is_fraud, dtype: float64


In [28]:
import sys
sys.path.append("..")

from src.data import load_raw, clean, time_split
from src.features import build_features

In [29]:
fresh = load_raw("../data/raw/card_transaction.v1.csv")
fresh = clean(fresh)
fresh = build_features(fresh)
fresh.shape

(3000000, 26)

In [30]:
for col in ["hour", "outside_us", "new_shop", "gap_minutes"]:
    print(col, fresh[col].sum(), df[col].sum())

hour 37400611 37400611
outside_us 21329 21329
new_shop 117755 117755
gap_minutes 3030321109.0 3030321109.0


In [31]:
del fresh

In [32]:
new = load_raw("../data/raw/card_transaction.v1.csv", start=3_000_000)
new = clean(new)
new = build_features(new)
new.shape

(3000000, 26)

In [33]:
last_user = df["user"].max()
print("last training user:", last_user)
print("first new user:", new["user"].min())

new = new[new["user"] > last_user]
print("new people:", new["user"].nunique())

last training user: 252
first new user: 252
new people: 253


In [34]:
cutoff = pd.Timestamp("2017-04-25 21:24:36")
new_before, new_after = time_split(new, cutoff)

print(new_after.shape)
print("frauds:", new_after["is_fraud"].sum())

(599848, 26)
frauds: 459


In [35]:
features = ["amount", "hour", "outside_us", "new_shop", "gap_minutes"]

X_new = new_after[features].fillna({"gap_minutes": 100000})
y_new = new_after["is_fraud"]

scores_new = model.predict_proba(X_new)[:, 1]
print("PR-AUC:", average_precision_score(y_new, scores_new))

top_new = scores_new.argsort()[::-1][:1000]
print("Frauds in top 1000:", y_new.iloc[top_new].sum())

PR-AUC: 0.07361906068253597
Frauds in top 1000: 142


In [36]:
new_after.groupby("use_chip")["is_fraud"].mean() * 10000

use_chip
Chip Transaction      9.711458
Online Transaction    1.500281
Swipe Transaction     3.387867
Name: is_fraud, dtype: float64

In [37]:
from xgboost import XGBClassifier
X_train = train[features].fillna({"gap_minutes": 100000})
X_valid = valid[features].fillna({"gap_minutes": 100000})

In [38]:
weight=(y_train==0).sum()/(y_train==1).sum()

xgb=XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="aucpr",
    early_stopping_rounds=50
)

In [39]:
xgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)
scores_xgb = xgb.predict_proba(X_valid)[:, 1]
print("PR-AUC:", average_precision_score(y_valid, scores_xgb))
top_x = scores_xgb.argsort()[::-1][:1000]
print("Frauds in top 1000:", y_valid.iloc[top_x].sum())

[0]	validation_0-aucpr:0.04095
[52]	validation_0-aucpr:0.03604
PR-AUC: 0.041940418707070624
Frauds in top 1000: 69


In [40]:
scores_train = xgb.predict_proba(X_train)[:, 1]
print("Train PR-AUC:", average_precision_score(y_train, scores_train))
print("Valid PR-AUC:", average_precision_score(y_valid, scores_xgb))
print("Best tree:", xgb.best_iteration)

Train PR-AUC: 0.03459355096078249
Valid PR-AUC: 0.041940418707070624
Best tree: 2


In [41]:
xgb2 = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="aucpr",
)

xgb2.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=25,
)

[0]	validation_0-aucpr:0.02876	validation_1-aucpr:0.04095
[25]	validation_0-aucpr:0.04628	validation_1-aucpr:0.02602
[50]	validation_0-aucpr:0.05141	validation_1-aucpr:0.03557
[75]	validation_0-aucpr:0.05728	validation_1-aucpr:0.05709
[100]	validation_0-aucpr:0.06505	validation_1-aucpr:0.09710
[125]	validation_0-aucpr:0.07035	validation_1-aucpr:0.09028
[150]	validation_0-aucpr:0.07215	validation_1-aucpr:0.08495
[175]	validation_0-aucpr:0.07355	validation_1-aucpr:0.08400
[200]	validation_0-aucpr:0.07544	validation_1-aucpr:0.08160
[225]	validation_0-aucpr:0.07700	validation_1-aucpr:0.07730
[250]	validation_0-aucpr:0.07879	validation_1-aucpr:0.07334
[275]	validation_0-aucpr:0.08156	validation_1-aucpr:0.07362
[299]	validation_0-aucpr:0.08334	validation_1-aucpr:0.07263


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [42]:
xgb3 = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    scale_pos_weight=weight,
    eval_metric="aucpr",
)
xgb3.fit(X_train, y_train)

scores_x3 = xgb3.predict_proba(X_valid)[:, 1]
print("Valid PR-AUC:", average_precision_score(y_valid, scores_x3))
top_x3 = scores_x3.argsort()[::-1][:1000]
print("Valid frauds in top 1000:", y_valid.iloc[top_x3].sum())

Valid PR-AUC: 0.09448767218229077
Valid frauds in top 1000: 131


In [43]:
X_new = new_after[features].fillna({"gap_minutes": 100000})

scores_new_x3 = xgb3.predict_proba(X_new)[:, 1]
print("New PR-AUC:", average_precision_score(y_new, scores_new_x3))

results_new = pd.DataFrame({"score": scores_new_x3, "is_fraud": y_new.values})
print("New frauds in top 1000:", results_new.sort_values("score", ascending=False).head(1000)["is_fraud"].sum())

New PR-AUC: 0.044724322780947935
New frauds in top 1000: 74


In [44]:
import sys
sys.path.append("..")
from src.data import load_raw, clean
from src.features import build_features

test = build_features(clean(load_raw("../data/raw/card_transaction.v1.csv", nrows=5000)))
test[["user", "card", "timestamp", "count_last_hour"]].head(15)

,user,card,timestamp,count_last_hour
0,0,0,2002-09-01 06:21:00,0.0
1,0,0,2002-09-01 06:42:00,1.0
2,0,0,2002-09-02 06:22:00,0.0
3,0,0,2002-09-02 17:45:00,0.0
4,0,0,2002-09-03 06:23:00,0.0
5,0,0,2002-09-03 13:53:00,0.0
6,0,0,2002-09-04 05:51:00,0.0
7,0,0,2002-09-04 06:09:00,1.0
8,0,0,2002-09-05 06:14:00,0.0
9,0,0,2002-09-05 09:35:00,0.0


In [45]:
test[["timestamp", "amount", "amount_vs_usual"]].head(10)

,timestamp,amount,amount_vs_usual
0,2002-09-01 06:21:00,134.09,1.000000
1,2002-09-01 06:42:00,38.48,0.286971
2,2002-09-02 06:22:00,120.34,1.394680
3,2002-09-02 17:45:00,128.95,1.320713
4,2002-09-03 06:23:00,104.71,0.992841
5,2002-09-03 13:53:00,86.19,0.818410
6,2002-09-04 05:51:00,93.84,0.918859
7,2002-09-04 06:09:00,123.50,1.223464
8,2002-09-05 06:14:00,61.72,0.594820
9,2002-09-05 09:35:00,57.10,0.576237


In [46]:
import sys
sys.path.append("..")
import pandas as pd
from src.data import load_raw, clean, time_split
from src.features import build_features, FEATURES

In [47]:
df = build_features(clean(load_raw("../data/raw/card_transaction.v1.csv")))
df.to_parquet("../data/processed/cleaned_data.parquet", index=False)

In [48]:
[c for c in FEATURES if c not in df.columns]

[]

In [49]:
cutoff = pd.Timestamp("2017-04-25 21:24:36")
train, valid = time_split(df, cutoff)

In [50]:
X_train = train[FEATURES].fillna({"gap_minutes": 100000})
y_train = train["is_fraud"]
X_valid = valid[FEATURES].fillna({"gap_minutes": 100000})
y_valid = valid["is_fraud"]

In [51]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

In [52]:
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
model.fit(X_train, y_train)
scores_valid = model.predict_proba(X_valid)[:, 1]
print("Valid PR-AUC:", average_precision_score(y_valid, scores_valid))
print("Valid frauds in top 1000:", top_1000(y_valid, scores_valid))



Valid PR-AUC: 0.09655848923739695
Valid frauds in top 1000: 199


In [53]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

def top_1000(y, scores):
    results = pd.DataFrame({"score": scores, "is_fraud": y.values})
    return results.sort_values("score", ascending=False).head(1000)["is_fraud"].sum()

# train
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
model.fit(X_train, y_train)

# validation (same people, after April 2017)
scores_valid = model.predict_proba(X_valid)[:, 1]
print("Valid PR-AUC:", average_precision_score(y_valid, scores_valid))
print("Valid frauds in top 1000:", top_1000(y_valid, scores_valid))

# new people (never seen, after April 2017)
new = build_features(clean(load_raw("../data/raw/card_transaction.v1.csv", start=3_000_000)))
new = new[new["user"] > df["user"].max()]
new_before, new_after = time_split(new, cutoff)

X_new = new_after[FEATURES].fillna({"gap_minutes": 100000})
y_new = new_after["is_fraud"]
scores_new = model.predict_proba(X_new)[:, 1]
print("New PR-AUC:", average_precision_score(y_new, scores_new))
print("New frauds in top 1000:", top_1000(y_new, scores_new))

Valid PR-AUC: 0.09655848923739695
Valid frauds in top 1000: 199
New PR-AUC: 0.0882796371888321
New frauds in top 1000: 177
